# 주제 1 baseline — Detector + SAM 자동 마스크 생성기

부경대학교 교내 컴퓨터비전 부트캠프 · 최종 프로젝트 baseline · 2026. 8. 7.

---

YOLO11 이 찾은 박스를 SAM 의 prompt 로 넘겨 **이름이 붙은 마스크**를 자동 생성하고, COCO 의 정답 마스크와 비교합니다.

| STEP | 하는 일 |
|---|---|
| 0 · 1 | 환경 준비 · COCO val2017 일부 내려받기 |
| 2 | YOLO11 + SAM 2.1 불러오기 |
| 3 | 전체 이미지에 탐지 → 분할 |
| 4 | 결과를 표로 모으기 |
| 5 | **정량 분석** — mask IoU 분포 |
| 6 | **실패 사례** 고르고 저장 |

**시작 전에** — `런타임 → 런타임 유형 변경 → T4 GPU`. 그리고 STEP 1 의 다운로드 셀을
가장 먼저 실행해 두세요. 받는 동안 아래를 읽으면 됩니다.

## STEP 0 · 환경 준비

In [ ]:
!pip install -q ultralytics

In [ ]:
# 그래프 한글 폰트 (실패해도 실습에는 지장 없음)
try:
    !apt-get install -qq -y fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    import matplotlib.pyplot as plt
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 준비 완료")
except Exception as e:
    print("폰트 설치 건너뜀:", e)

In [ ]:
import os, glob, json, urllib.request
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
np.random.seed(0)


def show(img_bgr, title=None, w=9):
    h = w * img_bgr.shape[0] / img_bgr.shape[1]
    plt.figure(figsize=(w, h))
    plt.imshow(img_bgr[:, :, ::-1]); plt.axis("off")
    if title:
        plt.title(title, fontsize=12)
    plt.show()


def overlay(img_bgr, mask, color=(60, 120, 240), alpha=0.5):
    layer = np.zeros_like(img_bgr)
    layer[np.asarray(mask).astype(bool)] = color
    return cv2.addWeighted(img_bgr, 1.0, layer, alpha, 0)


def mask_iou(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix = max(0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def qbin(series, q, names):
    """값을 분위 구간으로 나눈다. 값이 적어 구간이 줄어도 죽지 않는다."""
    b = pd.qcut(series, q, duplicates="drop")
    cats = list(b.cat.categories)
    lab = names[:len(cats)] if len(cats) <= len(names) else [str(c) for c in cats]
    return b.cat.rename_categories(lab)


print("준비 완료")

In [ ]:
os.makedirs("outputs", exist_ok=True)     # 결과 이미지를 여기에 저장한다
os.makedirs("data", exist_ok=True)
print(os.listdir("."))

## STEP 1 · 데이터 준비 — COCO val2017

정답 마스크가 있는 데이터라 **정량 비교가 가능**합니다.

이미지 전체(1 GB)를 받지 않고, 어노테이션만 받은 뒤 **필요한 이미지만 URL 로 내려받습니다.**
`N_IMAGES` 를 늘리면 그만큼 시간이 더 걸립니다.

In [ ]:
N_IMAGES = 60          # 분석할 이미지 수 — 처음에는 60장으로 시작하세요
CLASSES  = ["person", "car", "dog", "chair", "bottle"]   # 관심 클래스

# 어노테이션만 내려받는다 (약 241 MB, 1~2분)
if not os.path.exists("annotations/instances_val2017.json"):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o annotations_trainval2017.zip
print(os.path.exists("annotations/instances_val2017.json"))

In [ ]:
from pycocotools.coco import COCO

coco = COCO("annotations/instances_val2017.json")
cat_ids = coco.getCatIds(catNms=CLASSES)
id2name = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}

# 관심 클래스가 들어 있는 이미지만 모은다
img_ids = sorted({i for cid in cat_ids for i in coco.getImgIds(catIds=[cid])})[:N_IMAGES]
print("이미지", len(img_ids), "장   ·   클래스", CLASSES)

In [ ]:
# 이미지 파일을 하나씩 내려받는다 (60장 기준 30초 내외)
paths = {}
for k, iid in enumerate(img_ids):
    info = coco.loadImgs(iid)[0]
    p = os.path.join("data", info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(info["coco_url"], p)
    paths[iid] = p
    if (k + 1) % 20 == 0:
        print(k + 1, "/", len(img_ids))
print("내려받기 완료:", len(paths), "장")

데이터가 잘 받아졌는지 눈으로 확인합니다. **이 셀이 안 돌면 다음으로 넘어가지 마세요.**

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, iid in zip(axes, list(paths)[:4]):
    ax.imshow(cv2.imread(paths[iid])[:, :, ::-1]); ax.axis("off")
    ax.set_title(os.path.basename(paths[iid]), fontsize=10)
plt.tight_layout(); plt.show()

## STEP 2 · 모델 불러오기

3일차의 YOLO11 과 4일차의 SAM 2.1 을 그대로 씁니다.

In [ ]:
from ultralytics import YOLO, SAM

det = YOLO("yolo11n.pt")        # 더 정확하게 하고 싶으면 yolo11s / yolo11m
sam = SAM("sam2.1_b.pt")
print("클래스 수:", len(det.names))

## STEP 3 · 탐지 → 분할

이미지 한 장마다 ① YOLO 로 박스를 찾고 ② 그 박스를 SAM 에 넘겨 마스크를 만듭니다.
**이 셀이 이 노트북에서 가장 오래 걸립니다** (60장 기준 T4 에서 2 ~ 4분).

In [ ]:
CONF = 0.25          # ★ 바꿔 볼 값
IOU  = 0.70          # ★ 바꿔 볼 값

def detect_and_segment(path):
    """이미지 한 장 → [(클래스명, conf, box, mask), ...]"""
    d = det(path, conf=CONF, iou=IOU, verbose=False)[0]
    if len(d.boxes) == 0:
        return []
    boxes = d.boxes.xyxy.cpu().numpy().tolist()
    names = [d.names[int(c)] for c in d.boxes.cls]
    confs = [float(c) for c in d.boxes.conf]
    s = sam(path, bboxes=boxes, verbose=False)[0]
    masks = s.masks.data.cpu().numpy()
    return list(zip(names, confs, boxes, masks))


preds = {}
for k, (iid, p) in enumerate(paths.items()):
    preds[iid] = detect_and_segment(p)
    if (k + 1) % 10 == 0:
        print(k + 1, "/", len(paths), flush=True)
print("완료 — 총 예측", sum(len(v) for v in preds.values()), "건")

## STEP 4 · 결과를 표로 모으기

정답 마스크와 짝을 지어 IoU 를 계산합니다. 짝짓기 규칙은 3일차에 배운 그대로입니다 —
**같은 클래스이면서 박스 IoU 가 가장 큰 정답**을 짝으로 봅니다.

In [ ]:
def gt_of(iid):
    """정답 마스크 목록 [(클래스명, box, mask), ...]"""
    out = []
    for a in coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=cat_ids, iscrowd=False)):
        x, y, w, h = a["bbox"]
        out.append((id2name[a["category_id"]], [x, y, x + w, y + h], coco.annToMask(a)))
    return out


rows = []
for iid, items in preds.items():
    gts = gt_of(iid)
    used = set()
    for name, conf, box, mask in items:
        best, bi = 0.0, -1
        for gi, (gname, gbox, gmask) in enumerate(gts):
            if gname != name or gi in used:
                continue
            v = box_iou(box, gbox)
            if v > best:
                best, bi = v, gi
        m_iou = mask_iou(mask, gts[bi][2]) if bi >= 0 and best >= 0.5 else 0.0
        if bi >= 0 and best >= 0.5:
            used.add(bi)
        rows.append(dict(image=os.path.basename(paths[iid]), image_id=iid,
                         cls=name, conf=round(conf, 3),
                         box_iou=round(best, 3), mask_iou=round(m_iou, 3),
                         matched=bool(bi >= 0 and best >= 0.5),
                         area=int(np.asarray(mask).sum())))
    for gi, (gname, gbox, gmask) in enumerate(gts):
        if gi not in used:      # 아무 예측도 붙지 않은 정답 = 놓친 것
            rows.append(dict(image=os.path.basename(paths[iid]), image_id=iid,
                             cls=gname, conf=0.0, box_iou=0.0, mask_iou=0.0,
                             matched=False, area=int(gmask.sum())))

COLS = ["image", "image_id", "cls", "conf", "box_iou", "mask_iou", "matched", "area"]
df = pd.DataFrame(rows, columns=COLS)
print(df.shape)
df.head(10)

## STEP 5 · 정량 분석 ★

**여기서부터가 프로젝트입니다.** 아래는 출발점이고, 여러분의 질문을 하나 더 얹으세요.

In [ ]:
matched = df[df.matched]
print(f"전체 예측·정답 행    : {len(df)}")
print(f"짝이 맞은 것          : {len(matched)}  ({100*len(matched)/max(1,len(df)):.1f}%)")
print(f"mask IoU 평균         : {matched.mask_iou.mean():.3f}")
print(f"mask IoU 0.5 미만 비율: {100*(matched.mask_iou < 0.5).mean():.1f}%")
print()
print("클래스별")
print(matched.groupby("cls").agg(건수=("mask_iou", "size"),
                                 평균IoU=("mask_iou", "mean")).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(matched.mask_iou, bins=25, color="#156082", edgecolor="white")
axes[0].axvline(0.5, color="#E97132", lw=2, ls="--")
axes[0].set_xlabel("mask IoU"); axes[0].set_ylabel("건수")
axes[0].set_title("mask IoU 분포", fontsize=12)

g = matched.groupby("cls").mask_iou.mean().sort_values()
axes[1].barh(g.index, g.values, color="#156082")
axes[1].set_xlim(0, 1); axes[1].set_xlabel("평균 mask IoU")
axes[1].set_title("클래스별 평균", fontsize=12)
plt.tight_layout(); plt.show()

### 여기에 여러분의 질문을 하나 더

예시 — 아래 중 하나를 골라 셀을 추가하세요.

- 객체 **크기(area)** 와 IoU 는 관계가 있는가? (작은 물체가 더 나쁜가)
- **confidence** 가 높은 예측이 마스크도 좋은가?
- 실패를 **탐지 실패(놓침)** 와 **분할 실패(IoU 낮음)** 로 나누면 각각 몇 건인가?

In [ ]:
# 예시 — 실패를 두 종류로 나눠 세어 본다
missed = (~df.matched).sum()                       # 아예 못 찾은 것
bad_mask = ((df.matched) & (df.mask_iou < 0.5)).sum()   # 찾았지만 마스크가 나쁜 것
print(f"탐지 실패(놓침) : {missed} 건")
print(f"분할 실패(IoU<0.5): {bad_mask} 건")

# 예시 — 크기와 IoU 의 관계
plt.figure(figsize=(6.5, 4))
plt.scatter(matched.area, matched.mask_iou, s=14, alpha=0.6, color="#156082")
plt.xscale("log"); plt.xlabel("마스크 면적 (픽셀, 로그)"); plt.ylabel("mask IoU")
plt.title("작은 물체일수록 나쁜가?", fontsize=12)
plt.tight_layout(); plt.show()

## STEP 6 · 실패 사례 고르고 저장 ★

무작위로 고르지 말고 **숫자로 정렬해서** 뽑습니다.

In [ ]:
def draw_result(iid, save=None, title=""):
    img = cv2.imread(paths[iid])
    PAL = [(232, 96, 21), (50, 113, 233), (91, 125, 46), (163, 79, 122)]
    for k, (name, conf, box, mask) in enumerate(preds.get(iid, [])):
        img = overlay(img, mask, PAL[k % len(PAL)], 0.45)
        x1, y1 = int(box[0]), int(box[1])
        cv2.putText(img, f"{name} {conf:.2f}", (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 3)
        cv2.putText(img, f"{name} {conf:.2f}", (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, PAL[k % len(PAL)], 2)
    if save:
        cv2.imwrite(save, img)
    show(img, title)
    return img


worst = (matched.sort_values("mask_iou").drop_duplicates("image_id").head(3))
best = (matched.sort_values("mask_iou", ascending=False)
        .drop_duplicates("image_id").head(3))

for i, r in enumerate(best.itertuples(), 1):
    draw_result(r.image_id, f"outputs/success_{i}.png",
                f"성공 {i} — {r.cls}  mask IoU {r.mask_iou:.2f}")
for i, r in enumerate(worst.itertuples(), 1):
    draw_result(r.image_id, f"outputs/failure_{i}.png",
                f"실패 {i} — {r.cls}  mask IoU {r.mask_iou:.2f}")
print("저장:", sorted(os.listdir("outputs")))

### 실패 사례 캡션 (이 셀을 더블클릭해 채우세요)

| 파일 | 클래스 | IoU | 무엇이 문제였나 |
|---|---|---|---|
| failure_1.png | | | |
| failure_2.png | | | |
| failure_3.png | | | |

세 장의 **공통점**을 한 문장으로: 

## STEP 7 · 결과 이미지 내려받기

Colab 에서 `outputs/` 폴더를 통째로 압축해 내려받습니다.

In [ ]:
!zip -q -r outputs.zip outputs
try:
    from google.colab import files
    files.download("outputs.zip")
except Exception as e:
    print("Colab 이 아닙니다:", e)
print("저장된 이미지:", len(glob.glob("outputs/*.png")), "장")

## STEP 8 · 심화 트랙 (선택)

여기까지 여유 있게 끝냈다면 **`5일차_심화_finetuning_가이드.ipynb`** 를 여세요.
사전학습 모델을 그대로 쓰는 대신 **직접 fine-tuning 해서 같은 잣대로 비교**하는 트랙입니다.
평가에 **심화 10점**이 따로 배정되어 있습니다.

- 먼저 **위의 STEP 6 까지를 끝내 두어야** 합니다 — 기준선이 없으면 비교가 성립하지 않습니다
- T4 GPU 기준 30 epoch 에 10 ~ 15분
- **11:20 까지 여기에 도달하지 못했다면 심화는 포기하고 발표 준비로 넘어가세요**

심화는 fine-tuning 만 인정하는 것이 아닙니다. 새 지표를 직접 만들었거나, 데이터를 늘려
다시 쟀거나, prompt 전략을 체계적으로 비교했어도 같은 점수입니다.

---

## 제출 전 점검

- [ ] **런타임 → 런타임 다시 시작** 후 처음부터 끝까지 오류 없이 실행되는가
- [ ] `outputs/` 폴더에 성공 사례 3장 이상, 실패 사례 3장 이상
- [ ] 결과 이미지마다 캡션(파일명 또는 아래 마크다운 셀)이 붙어 있는가
- [ ] 표 또는 그래프가 하나 이상 있는가
- [ ] 아래 요약 다섯 줄을 채웠는가

## 결과 요약 (이 셀을 더블클릭해 직접 채우세요)

1. **무엇을 만들었나** —
2. **정량 결과** — (숫자 하나 이상)
3. **가장 잘 된 경우** —
4. **실패 유형과 개수** —
5. **시간이 더 있었다면** —
